# Domain Data And Fine-Tuning Readiness

Fine-tuning data is not the same thing as retrieval data. Documents are useful as evidence at answer time; supervised examples teach behavior. Provider guides such as [Mistral fine-tuning](https://docs.mistral.ai/capabilities/finetuning/text_vision_finetuning/) and [OpenAI fine-tuning data formatting](https://help.openai.com/en/articles/6811186-how-do-i-format-my-fine-tuning-data-for-the-openai-api) both expect structured examples, not only raw documents.


## Data Readiness Mental Model

Data readiness is less about having many files and more about assigning each record a job. Retrieval documents should be authoritative, chunkable, and traceable to a source. Behavior examples should show the input, the desired answer, and the style or refusal pattern to imitate. Evaluation requests should stay separate from both; they are the measuring stick for whether the strategy works.

This separation prevents two common forms of leakage. First, if evaluation questions become training examples, the model may appear better than it really is. Second, if style examples are treated as policy documents, the retriever may return old answers as if they were source-of-truth evidence.


## Definitions: Corpus, Examples, And Evaluation Set

A **retrieval corpus** is the collection of documents the system is allowed to search. It should contain source-of-truth material, not model answers. In this repo, the policy records are the corpus because they represent the evidence participants want the model to cite.

A **behavior example** is an input-output pair that demonstrates how an assistant should respond. It can teach tone, structure, or refusal behavior. It should not be treated as evidence unless it also points to an authoritative source.

An **evaluation set** is a group of held-out requests used to measure whether a strategy works. It should include easy cases, edge cases, and cases where the system should abstain. If evaluation examples are reused as training examples, the measurement becomes too optimistic.


### Exercise

Which records should stay held out: retrieval documents, style examples, or evaluation requests?


### Solution

Evaluation requests should stay held out so they can measure generalization and strategy quality.


## Visual Map: Data For Behavior vs Data For Evidence

The same raw domain material should not be pushed into every strategy. Retrieval needs source documents with metadata; supervised fine-tuning needs input-output examples; evaluation needs held-out requests.

**Prompt tuning as learned behavior control.** Source: [Hugging Face documentation images](https://huggingface.co/datasets/huggingface/documentation-images/blob/main/peft/prompt-tuning.png). This visual helps distinguish trainable prompt parameters from normal few-shot examples placed in a notebook prompt.

<img src="assets/hf-prompt-tuning.png" alt="Prompt tuning diagram" style="max-width: 760px; width: 100%; height: auto;">

---

The second adaptation visual shows a related soft-prompt approach, useful for comparing multiple ways to encode behavior.

**P-tuning as another soft-prompt method.** Source: [Hugging Face documentation images](https://huggingface.co/datasets/huggingface/documentation-images/blob/main/peft/p-tuning.png). This visual reinforces that fine-tuning-style behavior adaptation needs structured examples and evaluation, not a pile of raw policy text.

<img src="assets/hf-p-tuning.png" alt="P-tuning diagram" style="max-width: 760px; width: 100%; height: auto;">

---

The final visual in this section switches from behavior adaptation to retrieval data preparation.

**Embedding service flow.** Source: [Hugging Face cookbook images](https://huggingface.co/datasets/huggingface/cookbook-images/blob/main/automatic_embedding_tei_inference_endpoints.png). This visual connects data readiness to retrieval infrastructure: documents must become embeddings before they can support RAG.

<img src="assets/hf-cookbook-automatic-embedding-tei.png" alt="Automatic embedding workflow" style="max-width: 760px; width: 100%; height: auto;">


### Exercise

From the visuals, are prompt-tuning and P-tuning examples of retrieval or behavior adaptation?


### Solution

They are behavior-adaptation methods. Retrieval is about supplying external evidence at request time.


## Load Dependencies And Shared Data

The setup is split into small cells so each responsibility is visible: imports first, then `.env` loading and fixture loading.


In [ ]:
import json
import os
from pathlib import Path
from pprint import pprint
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

The notebooks share one small support-policy fixture. Keeping it shared makes the strategy comparisons fair: each notebook sees the same documents, requests, and style examples.


In [ ]:
# Load optional provider settings from .env without overwriting shell variables.
load_dotenv(dotenv_path=Path(".env"), override=False)

DATA_PATH = Path("data") / "strategy_cases.json"
with DATA_PATH.open(encoding="utf-8") as file:
    DATA = json.load(file)

# Keep each fixture role explicit so later cells do not mix evidence, examples, and evaluation.
DOCUMENTS = DATA["documents"]
REQUESTS = DATA["support_requests"]
STYLE_EXAMPLES = DATA["style_examples"]
SCENARIOS = DATA["strategy_scenarios"]

print(
    f"Loaded {len(DOCUMENTS)} documents, {len(REQUESTS)} requests, and {len(STYLE_EXAMPLES)} style examples."
)

## Configure The Provider

The notebooks use the OpenAI-compatible client style. Groq is tried first by default; Mistral is the fallback when the preferred provider has no key.


In [ ]:
class ProviderConfig(BaseModel):
    """Provider settings for one OpenAI-compatible chat endpoint."""

    provider: str
    model: str
    base_url: str
    api_key: str | None = None
    dry_run: bool = True

Provider selection is intentionally explicit. This makes fallback behavior easy to inspect before any model call happens.


In [ ]:
def provider_order(preferred_provider: str | None = None) -> list[str]:
    """Return the preferred provider followed by the other supported provider."""
    requested = (preferred_provider or os.getenv("LLM_PROVIDER") or "groq").lower()

    if requested == "groq":
        return ["groq", "mistral"]
    if requested == "mistral":
        return ["mistral", "groq"]

    # Unknown values keep dry mode readable instead of failing during setup.
    return [requested]

The loader returns dry mode when no key is available. Dry mode is part of the teaching design: notebooks stay runnable during review and in classrooms without credentials.


Each provider has a tiny environment-to-config adapter. Keeping this separate makes the fallback loader shorter and easier to read.


In [ ]:
def provider_config_from_env(provider_name: str) -> ProviderConfig | None:
    """Return live provider settings when that provider has an API key."""
    if provider_name == "groq" and os.getenv("GROQ_API_KEY"):
        return ProviderConfig(
            provider="groq",
            model=os.getenv("GROQ_MODEL", "openai/gpt-oss-20b"),
            base_url="https://api.groq.com/openai/v1",
            api_key=os.getenv("GROQ_API_KEY"),
            dry_run=False,
        )

    if provider_name == "mistral" and os.getenv("MISTRAL_API_KEY"):
        return ProviderConfig(
            provider="mistral",
            model=os.getenv("MISTRAL_MODEL", "mistral-small-latest"),
            base_url="https://api.mistral.ai/v1",
            api_key=os.getenv("MISTRAL_API_KEY"),
            dry_run=False,
        )

    return None

The loader checks providers in order and falls back to dry mode when no key is available.


In [ ]:
def load_provider_config(preferred_provider: str | None = None) -> ProviderConfig:
    """Load live provider settings, or return a deterministic dry-run config."""
    # Check providers in priority order so fallback behavior is deterministic.
    for provider_name in provider_order(preferred_provider):
        config = provider_config_from_env(provider_name)
        if config is not None:
            return config

    # No key was found, so keep the notebook runnable without network calls.
    fallback_model = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")
    return ProviderConfig(
        provider=provider_order(preferred_provider)[0],
        model=fallback_model,
        base_url="dry-run",
    )

In [ ]:
CONFIG = load_provider_config()
print(
    f"Provider: {CONFIG.provider} | model: {CONFIG.model} | dry run: {CONFIG.dry_run}"
)

if CONFIG.dry_run:
    print("Set GROQ_API_KEY or MISTRAL_API_KEY in .env to run live model calls.")

## Call The Model Or Use Dry Mode

The next cells isolate client creation, request assembly, and fallback behavior. That keeps provider plumbing separate from the lesson logic.


In [ ]:
def get_client(config: ProviderConfig) -> OpenAI | None:
    """Create a client only when a live API key is configured."""
    if config.dry_run or config.api_key is None:
        return None

    return OpenAI(api_key=config.api_key, base_url=config.base_url)

Request parameters are built in one small helper so response formatting can be added only where a notebook needs structured JSON.


In [ ]:
def completion_kwargs(
    messages: list[dict[str, str]],
    *,
    temperature: float,
    response_format: dict[str, str] | None,
) -> dict[str, Any]:
    """Build provider request arguments for chat completions."""
    kwargs: dict[str, Any] = {
        "model": CONFIG.model,
        "messages": messages,
        "temperature": temperature,
    }

    # JSON mode is optional because not every provider/model combination needs it.
    if response_format is not None:
        kwargs["response_format"] = response_format

    return kwargs

`call_model` is the single gateway used by lesson cells. If live credentials are missing or a live call fails, the notebook returns a deterministic teaching response.


Live completion execution is separated from dry-mode routing so the failure handling is easy to follow.


In [ ]:
def live_completion_content(
    messages: list[dict[str, str]],
    *,
    temperature: float,
    response_format: dict[str, str] | None,
) -> str:
    """Run the live chat completion and return message content."""
    client = get_client(CONFIG)
    assert client is not None

    completion = client.chat.completions.create(
        **completion_kwargs(
            messages, temperature=temperature, response_format=response_format
        )
    )
    return completion.choices[0].message.content or ""

`call_model` now only decides between dry output, live output, and live-call fallback.


In [ ]:
def call_model(
    messages: list[dict[str, str]],
    *,
    dry_response: str,
    temperature: float = 0.2,
    response_format: dict[str, str] | None = None,
) -> str:
    """Call the configured provider, or return deterministic dry output."""
    # Dry mode is the expected path for participants without provider credentials.
    if CONFIG.dry_run:
        return dry_response

    try:
        return live_completion_content(
            messages, temperature=temperature, response_format=response_format
        )
    except Exception as error:
        print(f"Live provider call failed: {error}")
        print("Using deterministic dry response so the notebook remains runnable.")
        return dry_response

## Classify Data Roles

We keep three roles separate to avoid leakage:

- retrieval corpus: source documents for RAG;
- behavior examples: demonstrations that could become few-shot or fine-tuning records;
- evaluation requests: held-out checks for strategy decisions.


In [ ]:
inventory = [
    {"name": "policy documents", "count": len(DOCUMENTS), "role": "retrieval_corpus"},
    {
        "name": "style examples",
        "count": len(STYLE_EXAMPLES),
        "role": "behavior_examples",
    },
    {"name": "support requests", "count": len(REQUESTS), "role": "evaluation_requests"},
]
pprint(inventory)

In [ ]:
# Exercise: count the records available for retrieval.
# retrieval_record_count = ...


In [ ]:
# Solution
retrieval_record_count = len(DOCUMENTS)
print(retrieval_record_count)

In [ ]:
# Exercise: write the role used for style examples.
# style_examples_role = ...


In [ ]:
# Solution
style_examples_role = "behavior_examples"
print(style_examples_role)

## Convert Style Examples Into Chat Records

The same message shape works for few-shot prompting today and can be adapted into future supervised fine-tuning JSONL records.


## What Makes An Example Useful?

A useful behavior example is specific enough to teach the desired response pattern without smuggling in unstable facts. Good examples usually include the user intent, the desired tone, and the exact assistant behavior that should be repeated. Weak examples are often too broad: “be helpful” does not teach the model when to refuse, when to ask a clarifying question, or when to cite policy.

For future fine-tuning, examples also need consistency. If one refund example says to ask for invoice numbers and another says to approve refunds immediately, the dataset teaches conflicting behavior. The goal in this notebook is not to create a production training set, but to learn how to inspect records before they become prompts, retrieval documents, or supervised examples.


## Example Quality Checklist

Before a record becomes a few-shot example or future training example, check four things. First, the user message should be realistic enough to match a future request. Second, the assistant answer should demonstrate the exact behavior you want repeated. Third, the record should have labels such as intent or tone so the dataset can be audited. Fourth, the answer should avoid hardcoding unstable facts unless those facts are also present in a source document.

For example, “Reply politely” is too vague. “Ask for the invoice number and both payment references, then mention that finance replies within three business days” is more useful because it teaches structure and tone. If the three-day detail changes often, that part belongs in retrieval evidence rather than in a behavior-only example.


Define small validation models before converting any examples. The schema catches missing roles or empty messages early.


In [ ]:
class ChatMessage(BaseModel):
    """A minimal chat message record for fine-tuning-style examples."""

    role: str = Field(pattern="^(system|user|assistant)$")
    content: str = Field(min_length=1)


class TrainingExample(BaseModel):
    """Provider-neutral training example with labels for auditability."""

    messages: list[ChatMessage]
    intent: str
    tone: str

Convert each style example into the same chat-message shape used by modern supervised fine-tuning APIs.


In [ ]:
def to_training_example(example: dict[str, str]) -> TrainingExample:
    """Convert one style example into a chat-shaped record."""
    return TrainingExample(
        intent=example["intent"],
        tone=example["tone"],
        messages=[
            ChatMessage(
                role="system",
                content="Answer as a support assistant using the requested tone.",
            ),
            ChatMessage(role="user", content=example["user"]),
            ChatMessage(role="assistant", content=example["assistant"]),
        ],
    )

Build the converted records and inspect one example before using them in readiness checks.


In [ ]:
training_examples = [to_training_example(example) for example in STYLE_EXAMPLES]
pprint(training_examples[0].model_dump())

In [ ]:
# Exercise: count assistant messages in the first converted example.
# assistant_message_count = ...


In [ ]:
# Solution
assistant_message_count = sum(
    message.role == "assistant" for message in training_examples[0].messages
)
print(assistant_message_count)

## Ask The Model To Review Readiness

The model can help critique whether examples are ready for future fine-tuning. This is a live provider call when credentials exist; dry mode returns a deterministic review.


Serialize the validated examples so the provider reviews the same structure we inspected.


In [ ]:
readiness_prompt = json.dumps(
    [example.model_dump() for example in training_examples], indent=2
)

readiness_messages = [
    {
        "role": "system",
        "content": "You review supervised fine-tuning datasets. Return JSON with strengths, gaps, and next_steps.",
    },
    {
        "role": "user",
        "content": f"Review these examples for future fine-tuning readiness:\n{readiness_prompt}",
    },
]

Keep the dry review deterministic so the lesson can run without credentials.


In [ ]:
dry_readiness = json.dumps(
    {
        "strengths": [
            "examples are chat-shaped",
            "intent and tone labels are explicit",
        ],
        "gaps": [
            "too few examples",
            "billing intent is missing",
            "no held-out style evaluation set",
        ],
        "next_steps": [
            "collect more examples per intent",
            "separate training and evaluation records",
        ],
    },
    indent=2,
)

Call the provider or dry-mode fallback, then parse the JSON review.


In [ ]:
raw_review = call_model(
    readiness_messages,
    dry_response=dry_readiness,
    response_format={"type": "json_object"},
)
print(raw_review)

try:
    parsed_review = json.loads(raw_review)
    pprint(parsed_review)
except json.JSONDecodeError:
    print("The provider returned non-JSON text; refine the prompt or parse manually.")

In [ ]:
role_map = {item["name"]: item["role"] for item in inventory}
pprint(role_map)